In [42]:
#Importing necessary libraries
import sqlite3
import pandas as pd
import os

In [43]:
#Connect to the database file, send SQL queries to the database and extract the results.

mydbpath = '/content/assignment.db' #Path to the database file
#Establishing a connection to the database
if os.path.exists(mydbpath):
    try:
        conn = sqlite3.connect(mydbpath)
        cursor = conn.cursor()
        print("Database connection successful.")
    except sqlite3.Error as e:
        print(f"SQLite error: {e}")
else:
    print("Database file does not exist. Connection aborted.")

country_data = cursor.execute("SELECT * FROM country").fetchall()
bill_data = cursor.execute("SELECT * FROM bill").fetchall()
partner_data = cursor.execute("SELECT * FROM partner").fetchall()
customer_data = cursor.execute("SELECT * FROM customer").fetchall()

# Printing the results
print("Country Table:")
for row in country_data:
    print(row)

print("\nBill Table:")
for row in bill_data:
    print(row)

print("\nPartner Table:")
for row in partner_data:
    print(row)

print("\nCustomer Table:")
for row in customer_data:
    print(row)

Database connection successful.
Country Table:
('DE', 'Germany')
('FR', 'France')
('IT', 'Italy')
('ES', 'Spain')
('NL', 'Netherlands')
('BE', 'Belgium')
('AT', 'Austria')
('FI', 'Finland')
('PT', 'Portugal')
('IE', 'Ireland')

Bill Table:
('BL0001', 'B00001', '01-09-2025', 1200)
('BL0002', 'B00002', '02-09-2025', 2500)
('BL0003', 'B00003', '03-09-2025', 800)
('BL0004', 'B00004', '04-09-2025', 4500)
('BL0005', 'B00005', '05-09-2025', 1600)
('BL0006', 'B00006', '06-09-2025', 2100)
('BL0007', 'B00001', '07-09-2025', 900)
('BL0008', 'B00002', '08-09-2025', 3500)
('BL0009', 'B00003', '09-09-2025', 2200)
('BL0010', 'B00004', '10-09-2025', 1300)
('BL0011', 'B00005', '11-09-2025', 1900)
('BL0012', 'B00006', '12-09-2025', 2750)

Partner Table:
('P00001', 'AutoParts GmbH', 'DE', 'contact@ap.de', '+491234567890')
('P00002', 'MaisonElectro SA', 'FR', 'info@me.fr', '+331234567891')
('P00003', 'Mobili Italia SRL', 'IT', 'sales@mi.it', '+391234567892')
('P00004', 'Iberica Foods', 'ES', 'contact@ib.e

In [44]:
#Read the content of the customer relation (table) into Pandas DataFrame.

customer_df = pd.read_sql_query("SELECT * FROM customer", conn)
customer_df

,customer_id,first_name,last_name,email,customer_phone,street,house_number,city,country_code
0,C00001,Anna,Müller,anna.mueller@gmail.com,+491111111111,Berliner Strasse,10A,Berlin,DE
1,C00002,Pierre,Dubois,pierre.dubois@orange.fr,+331111111111,Rue de Rivoli,5B,Paris,FR
2,C00003,Luca,Rossi,luca.rossi@libero.it,+391111111111,Via Roma,22,Rome,IT
3,C00004,Maria,Lopez,maria.lopez@hotmail.es,+341111111111,Calle Mayor,12,Madrid,ES
4,C00005,Jan,Jansen,jan.jansen@kpn.nl,+311111111111,Damrak,30,Amsterdam,NL
5,C00006,Sophie,Vermeer,sophie.vermeer@ziggo.nl,+31611111112,Keizersgracht,12,Amsterdam,NL
6,C00007,Paul,Schmidt,paul.schmidt@yahoo.de,+491111111112,Königsallee,100,Düsseldorf,DE
7,C00008,Laura,Meyer,laura.meyer@web.de,+491111111113,Hauptstrasse,3,Munich,DE
8,C00009,Isabelle,Martin,isabelle.martin@sfr.fr,+331111111113,Boulevard St.,44,Lyon,FR
9,C00010,Carlos,Garcia,carlos.garcia@telefonica.es,+341111111112,Gran Via,50,Barcelona,ES


In [45]:
#Using jaro similarity function that compares two tuples and reports the customer names with similarity > 0.7.

def jaro(s, t):
    '''Jaro similarity between two strings.'''
    s_len = len(s)
    t_len = len(t)

    if s_len == 0 and t_len == 0:
        return 1

    match_distance = (max(s_len, t_len) // 2) - 1

    s_matches = [False] * s_len
    t_matches = [False] * t_len

    matches = 0
    transpositions = 0

    for i in range(s_len):
        start = max(0, i - match_distance)
        end = min(i + match_distance + 1, t_len)

        for j in range(start, end):
            if t_matches[j]:
                continue
            if s[i] != t[j]:
                continue
            s_matches[i] = True
            t_matches[j] = True
            matches += 1
            break

    if matches == 0:
        return 0

    k = 0
    for i in range(s_len):
        if not s_matches[i]:
            continue
        while not t_matches[k]:
            k += 1
        if s[i] != t[k]:
            transpositions += 1
        k += 1

    return ((matches / s_len) +
            (matches / t_len) +
            ((matches - transpositions / 2) / matches)) / 3

jaro(customer_df['first_name'][0], customer_df['first_name'][19]) #Comparing 'Anna' and 'Sanna' from customer

0.7833333333333333

In [46]:
#Using jaccard similarity function that compares two records and reports the customers with similarity > 0.7.

def jaccard(s, t):
    '''Compute Jaccard similarity between two strings.'''
    set_s = set(s.lower())
    set_t = set(t.lower())
    intersection = set_s & set_t
    union = set_s | set_t
    if not union:
        return 0
    return len(intersection) / len(union)


def compare_customers_jaccard(df, threshold):
    similar_pairs = []
    string_columns = df.select_dtypes(include='object').columns

    for i in range(len(df)):
        for j in range(i + 1, len(df)):
            row1 = df.iloc[i]
            row2 = df.iloc[j]
            scores = []

            for col in string_columns:
                val1 = str(row1[col])
                val2 = str(row2[col])
                score = jaccard(val1, val2)
                scores.append(score)

            avg_score = sum(scores) / len(scores)
            if avg_score > threshold:
                similar_pairs.append({
                    'Customer 1 ID': row1['customer_id'],
                    'Customer 2 ID': row2['customer_id'],
                    'Average Similarity': round(avg_score, 3)
                })

    return pd.DataFrame(similar_pairs)


similar_customers = compare_customers_jaccard(customer_df, threshold=0.7)
print(similar_customers)

  Customer 1 ID Customer 2 ID  Average Similarity
0        C00006        C00021               0.742
1        C00007        C00022               0.853
2        C00008        C00023               0.732


In [47]:
#Closing the database connection
if conn:
    conn.close()
    print("Database connection closed.")

Database connection closed.
